In [3]:
cd /home/nampv1/projects/vnpost_asr

/home/nampv1/projects/vnpost_asr


In [4]:
# !mkdir prepare_data
# !mkdir prepare_data/vivos_dataset

In [113]:
%%writefile prepare_data/vivos_dataset/vivos.py
import os
import datasets


# _CITATION = """\
# @misc{vivos,
#   title = {VIVOS: Vietnamese Speech Corpus},
#   year = {2016},
#   publisher = {Viettel},
# }
# """

# _DESCRIPTION = """\
# VIVOS là một tập dữ liệu tiếng Việt dành cho Automatic Speech Recognition (ASR).
# Bao gồm train/test split với audio .wav, transcript (prompts.txt) và metadata (genders.txt).
# """

# _HOMEPAGE = "https://ailab.hcmus.edu.vn/vivos/"
# _LICENSE = "CC BY-SA 4.0"


class Vivos(datasets.GeneratorBasedBuilder):
    VERSION = datasets.Version("1.0.0")

    def _info(self):
        return datasets.DatasetInfo(
            # description=_DESCRIPTION,
            features=datasets.Features({
                "sample_id": datasets.Value("uint32"),
                "audio": datasets.Audio(sampling_rate=16000),
                "filename": datasets.Value("string"),
                "speaker_id": datasets.Value("string"),
                "gender": datasets.ClassLabel(names=["m", "f"]),
                # "sentence_id": datasets.Value("string"),
                "text": datasets.Value("string"),
            }),
            supervised_keys=("audio", "text"),
            # homepage=_HOMEPAGE,
            # license=_LICENSE,
            # citation=_CITATION,
        )

    def _split_generators(self, dl_manager):
        data_dir = self.config.data_dir
        return [
            datasets.SplitGenerator(
                name=datasets.Split.TRAIN,
                gen_kwargs={
                    "split_name": "train",
                    "data_dir": os.path.join(data_dir, "train"),
                },
            ),
            datasets.SplitGenerator(
                name=datasets.Split.TEST,
                gen_kwargs={
                    "split_name": "test",
                    "data_dir": os.path.join(data_dir, "test"),
                },
            ),
        ]

    def _generate_examples(self, split_name, data_dir):
        # Đọc genders.txt
        genders_path = os.path.join(data_dir, "genders.txt")
        genders = {}
        with open(genders_path, encoding="utf-8") as f:
            for line in f:
                spk, g = line.strip().split()
                genders[spk] = g

        # Đọc prompts.txt
        prompts_path = os.path.join(data_dir, "prompts.txt")
        prompts = {}
        with open(prompts_path, encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split(maxsplit=1)
                if len(parts) == 2:
                    sent_id, text = parts
                    prompts[sent_id] = text

        # Traverse waves/
        wav_dir = os.path.join(data_dir, "waves")
        idx = 0
        for spk in sorted(os.listdir(wav_dir)):
            spk_dir = os.path.join(wav_dir, spk)
            if not os.path.isdir(spk_dir):
                continue
            gender = genders.get(spk, None)
            for wav_file in sorted(os.listdir(spk_dir)):
                if not wav_file.endswith(".wav"):
                    continue
                sentence_id = os.path.splitext(wav_file)[0]
                text = prompts.get(sentence_id, "")
                path = os.path.join(spk_dir, wav_file)
                yield idx, {
                    "sample_id": idx,
                    "audio": path,
                    "filename": wav_file,
                    "speaker_id": spk,
                    "gender": gender,
                    # "sentence_id": sentence_id,
                    "text": text,
                }
                idx += 1


Writing prepare_data/vivos_dataset/vivos.py


In [115]:
%%writefile prepare_data/vivos_dataset/__init__.py
#

Writing prepare_data/vivos_dataset/__init__.py


In [125]:
local_raw_data_dir_vivos = "/media/nampv1/hdd/data/VIVOS/raw/vivos"
hf_local_raw_data_dir_vivos = "/media/nampv1/hdd/data/VIVOS/raw/hf"

In [120]:
import os
local_raw_data_dir_vivos_relpath = os.path.relpath(local_raw_data_dir_vivos, "prepare_data/vivos_dataset")
local_raw_data_dir_vivos_relpath

'../../../../../../media/nampv1/hdd/data/VIVOS/raw/vivos'

In [5]:
from datasets import load_dataset, DatasetDict

def create_hf_ds(dataset_script_path: str, 
    data_dir: str, 
    save_dir: str = None, 
    streaming: bool = False) -> DatasetDict:
    """
    Create Hugging Face dataset from a local dataset loading script.

    Args:
        dataset_script_path (str): path to dataset.py (e.g., "prepare_data/vivos_dataset/vivos.py")
        data_dir (str): path to raw dataset directory (e.g., local_raw_data_dir_vivos)
        save_dir (str, optional): if given, will save the dataset to this directory
        streaming (bool): whether to use streaming mode (avoid loading full dataset into RAM)

    Returns:
        DatasetDict
    """
    print(f"Loading dataset using script={dataset_script_path}, data_dir={data_dir} ...")

    ds = load_dataset(
        path=dataset_script_path,
        data_dir=data_dir,
        trust_remote_code=True,  # needed if dataset.py uses custom code
        streaming=streaming,
    )

    if not streaming:
        print(ds)
        print("Example sample:", ds["train"][0])

        if save_dir:
            print(f"Saving dataset to disk at {save_dir} ...")
            ds.save_to_disk(save_dir)

    return ds


In [6]:
local_raw_data_dir_vivos = "/media/nampv1/hdd/data/VIVOS/raw/vivos"
dataset_script = "prepare_data/vivos_dataset/vivos.py"
hf_out_dir = "/media/nampv1/hdd/data/VIVOS/raw/hf"

dataset = create_hf_ds(dataset_script, data_dir = local_raw_data_dir_vivos)


Loading dataset using script=prepare_data/vivos_dataset/vivos.py, data_dir=/media/nampv1/hdd/data/VIVOS/raw/vivos ...
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'audio', 'filename', 'speaker_id', 'gender', 'text'],
        num_rows: 11660
    })
    test: Dataset({
        features: ['sample_id', 'audio', 'filename', 'speaker_id', 'gender', 'text'],
        num_rows: 760
    })
})
Example sample: {'sample_id': 0, 'audio': {'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK01/VIVOSSPK01_R001.wav', 'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
       -4.48608398e-03, -5.15747070e-03, -3.69262695e-03], shape=(48000,)), 'sampling_rate': 16000}, 'filename': 'VIVOSSPK01_R001.wav', 'speaker_id': 'VIVOSSPK01', 'gender': 1, 'text': 'KHÁCH SẠN'}


In [7]:
dataset["train"][0]

{'sample_id': 0,
 'audio': {'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK01/VIVOSSPK01_R001.wav',
  'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
         -4.48608398e-03, -5.15747070e-03, -3.69262695e-03], shape=(48000,)),
  'sampling_rate': 16000},
 'filename': 'VIVOSSPK01_R001.wav',
 'speaker_id': 'VIVOSSPK01',
 'gender': 1,
 'text': 'KHÁCH SẠN'}

In [8]:
len(dataset['train'])

11660

In [9]:
from src.utils.audio_utils import listen_audio, show_sample_by_filename, listen_audio

In [10]:
from pprint import pprint

In [11]:
for idx in range(1000, 1010):
    pprint(dataset["train"][idx])
    listen_audio(dataset["train"][idx]['audio']['array'])

{'audio': {'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
        1.67846680e-03,  2.07519531e-03,  2.56347656e-03], shape=(58000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R001.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R001.wav',
 'gender': 1,
 'sample_id': 1000,
 'speaker_id': 'VIVOSSPK05',
 'text': 'ĐỂ TỰ MÌNH TÌM RA CÁC CHÂN LÝ CỦA ĐẠO PHẬT'}


{'audio': {'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
       -8.54492188e-04, -1.37329102e-03, -1.64794922e-03], shape=(43000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R002.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R002.wav',
 'gender': 1,
 'sample_id': 1001,
 'speaker_id': 'VIVOSSPK05',
 'text': 'ĐỰNG TOÀN BỘ SỐ TRANH BỊ THẤT LẠC'}


{'audio': {'array': array([0.00000000e+00, 3.05175781e-05, 0.00000000e+00, ...,
       4.11987305e-03, 3.05175781e-03, 2.28881836e-03], shape=(34000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R003.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R003.wav',
 'gender': 1,
 'sample_id': 1002,
 'speaker_id': 'VIVOSSPK05',
 'text': 'MÀ VÌ TÔI LÀ NGƯỜI NÓNG VỘI'}


{'audio': {'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
       -1.77001953e-03, -1.61743164e-03, -1.12915039e-03], shape=(57000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R004.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R004.wav',
 'gender': 1,
 'sample_id': 1003,
 'speaker_id': 'VIVOSSPK05',
 'text': 'CHỈ CÓ NGƯỜI TRỰC TIẾP SẢN XUẤT NÔNG NGHIỆP'}


{'audio': {'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
       -7.32421875e-04, -9.15527344e-04, -1.06811523e-03], shape=(51000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R005.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R005.wav',
 'gender': 1,
 'sample_id': 1004,
 'speaker_id': 'VIVOSSPK05',
 'text': 'VÌ PHẢI THƯỜNG XUYÊN TIẾP XÚC VỚI LỬA'}


{'audio': {'array': array([ 0.00000000e+00,  3.05175781e-05, -3.05175781e-05, ...,
       -2.04467773e-03, -2.28881836e-03, -1.77001953e-03], shape=(49000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R006.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R006.wav',
 'gender': 1,
 'sample_id': 1005,
 'speaker_id': 'VIVOSSPK05',
 'text': 'BỖNG NHIÊN LŨ BÒ RỐNG LÊN THẢM THIẾT'}


{'audio': {'array': array([0.00000000e+00, 3.05175781e-05, 0.00000000e+00, ...,
       1.22070312e-03, 7.62939453e-04, 8.85009766e-04], shape=(70000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R007.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R007.wav',
 'gender': 1,
 'sample_id': 1006,
 'speaker_id': 'VIVOSSPK05',
 'text': 'THÌ NGHỆ SĨ CŨNG ĐẠT ĐƯỢC MỤC TIÊU QUẢNG BÁ TÊN TUỔI'}


{'audio': {'array': array([ 0.00000000e+00,  3.05175781e-05,  0.00000000e+00, ...,
       -4.51660156e-03, -4.54711914e-03, -4.45556641e-03], shape=(68000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R008.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R008.wav',
 'gender': 1,
 'sample_id': 1007,
 'speaker_id': 'VIVOSSPK05',
 'text': 'ANH BẮT ĐẦU CHỜ Ở KÍ TÚC XÁ ĐỂ ĐƯA EM ĐẾN TRƯỜNG'}


{'audio': {'array': array([ 0.00000000e+00,  0.00000000e+00, -3.05175781e-05, ...,
        2.31933594e-03,  2.62451172e-03,  3.05175781e-03], shape=(73000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R009.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R009.wav',
 'gender': 1,
 'sample_id': 1008,
 'speaker_id': 'VIVOSSPK05',
 'text': 'ĐẶC TRƯNG BỞI NHỮNG CƠN ĐAU NHƯ DAO ĐÂM Ở VÙNG CỔ TRÊN'}


{'audio': {'array': array([0.00000000e+00, 3.05175781e-05, 0.00000000e+00, ...,
       3.96728516e-04, 8.23974609e-04, 4.57763672e-04], shape=(50000,)),
           'path': '/media/nampv1/hdd/data/VIVOS/raw/vivos/train/waves/VIVOSSPK05/VIVOSSPK05_R010.wav',
           'sampling_rate': 16000},
 'filename': 'VIVOSSPK05_R010.wav',
 'gender': 1,
 'sample_id': 1009,
 'speaker_id': 'VIVOSSPK05',
 'text': 'GÁNH NẶNG ĐÒN GÁNH OẰN XUỐNG'}


In [ ]:
local_raw_data_dir_vivos = "/media/nampv1/hdd/data/VIVOS/raw/vivos"
hf_local_raw_data_dir_vivos = "/media/nampv1/hdd/data/VIVOS/raw/hf"

In [126]:
dataset.save_to_disk(hf_local_raw_data_dir_vivos)

Saving the dataset (1/1 shards): 100%|██████████| 760/760 [00:00<00:00, 1284.81 examples/s]


In [25]:
from src.utils.audio_utils import show_sample_by_filename
from src.utils.utils import load_dict_from_json

In [42]:
filename="VIVOSSPK01_R002.wav"
split_dataset = dataset['train']
split_filename2sid = load_dict_from_json("/media/nampv1/hdd/data/VIVOS/processed/all_filename2sid.json")['train']

In [43]:
show_sample_by_filename(filename, split_dataset, split_filename2sid)

sample_id: 1
filename: VIVOSSPK01_R002.wav
speaker_id: VIVOSSPK01
gender: 1
text: CHỈ BẰNG CÁCH LUÔN NỖ LỰC THÌ CUỐI CÙNG BẠN MỚI ĐƯỢC ĐỀN ĐÁP


In [23]:
!python prepare_data1.py \
    --config_path="configs/12092025/vivos__openai_whisper-large-v3-turbo.yaml" \
    exp_manager.exp_variant="exp0" \
    data.use_existing_hfds=false \
    data.root_data_dir="/media/nampv1/hdd/data/VIVOS/" \
    data.hf_raw_data_dir="/media/nampv1/hdd/data/VIVOS/raw/hf" \
    data.prepared_data_dir="/media/nampv1/hdd/data/VIVOS/exps/toy2" \
    data.do_shard_for_feature_computation=False \
    data.num_proc=1 \
    data.subset_ratio=0.01

exp_manager:
  prj_name: vnpost_asr_ft
  exp_name: vivos__openai_whisper-large-v3-turbo
  exp_variant: exp0
  exp_notes: ''
  seed: 202508
  task_name: vi_asr
  dataset_name: null
  model_name: openai/whisper-large-v3-turbo
  phase_name: eval
  print_cfg: true
  exps_dir: exps
  print_model: true
  print_processor: false
  print_trainable_parameters: true
  print_parameter_datatypes: true
  print_peft_config: true
  print_device: true
  print_training_args: false
  exp_tracking_tool: wandb
  wandb:
    project: vnpost_asr_ft
    log_artifact: false
data:
  continue_prep: false
  prepared_data_dir: /media/nampv1/hdd/data/VIVOS/exps/toy2
  root_data_dir: /media/nampv1/hdd/data/VIVOS/
  dataset_source_dir: /media/nampv1/hdd/data/VIVOS/raw/vivos
  hf_raw_data_dir: /media/nampv1/hdd/data/VIVOS/raw/hf
  dataset_script_path: prepare_data/vivos_hfds/vivos.py
  use_existing_hfds: false
  save_hfds: false
  do_shard_for_feature_computation: false
  streaming: false
  columns_to_retain:
  - sampl

In [17]:
!python prepare_data.py \
    --config_path="configs/12092025/vivos__openai_whisper-large-v3-turbo.yaml" \
    exp_manager.exp_variant="exp0" \
    data.root_data_dir="/media/nampv1/hdd/data/VIVOS/" \
    data.raw_data_dir="/media/nampv1/hdd/data/VIVOS/raw/hf" \
    data.do_shard_for_feature_computation=False \
    data.num_proc=1

Traceback (most recent call last):
  File "/home/nampv1/projects/vnpost_asr/prepare_data.py", line 11, in <module>
    from src.utils.model_utils import load_whisper_model, load_processor
  File "/home/nampv1/projects/vnpost_asr/src/utils/model_utils.py", line 5, in <module>
    from transformers import (
  File "<frozen importlib._bootstrap>", line 1231, in _handle_fromlist
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/transformers/utils/import_utils.py", line 2302, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/transformers/utils/import_utils.py", line 2330, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/nampv1/anaconda3/envs/asr/lib/python3.11/importlib/__init__.py", line 126, in import_mod